# Lab 25: AI Literacy & Generative Tools — Diagnostic Lab (ECON 5200)
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 45 min Portfolio + 45 min RAG Pipeline

---

**Format:** Part 1 builds a portfolio site with Claude Code (faster-paced than 3916). Part 2 presents a **deliberately broken RAG pipeline** that you must diagnose, fix, and extend.

**Learning Objectives:**
- Deploy a professional portfolio website using AI-assisted development
- Diagnose configuration errors in a Retrieval-Augmented Generation pipeline
- Compare RAG-grounded answers to direct LLM answers on economic questions
- Evaluate retrieval quality with precision and relevance metrics

**Tools:** Claude Code CLI, Next.js, Vercel, LangChain, ChromaDB, OpenAI API

**Time estimate:** ~90 minutes total

---

---

## Part 1: Portfolio Site (45 min)

Build and deploy a professional portfolio site with Claude Code. You are expected to move quickly — the setup and scaffold steps are compressed.

### Setup (5 min)

```bash
# Install Claude Code CLI (if not already installed)
npm install -g @anthropic-ai/claude-code
claude login
claude skill install /build-portfolio-site
```

Create a [Vercel account](https://vercel.com) via GitHub SSO if you do not have one.

### Scaffold & Customize (25 min)

```bash
mkdir econ-portfolio && cd econ-portfolio
claude
```

In the Claude Code session, provide your resume content, style preferences (browse [stitch.withgoogle.com](https://stitch.withgoogle.com) for reference), and at least **3 projects** from your coursework. Your prompt should include:

- Name, program, professional bio
- Style reference (colors, layout preference)
- Project titles, descriptions, GitHub links, and technologies
- A custom section (choose: Data Projects gallery, Research Interests, or Skills display)

### Deploy (10 min)

```bash
git init && git add . && git commit -m "Initial portfolio"
gh repo create econ-portfolio --public --push
```

Import to Vercel at [vercel.com/new](https://vercel.com/new). Test on desktop and mobile.

### Reflection (5 min)

In a markdown cell below, briefly answer:
1. What prompts worked best? What required iteration?
2. What did you have to fix that the AI got wrong?
3. How does this compare to writing the site from scratch?

**Your live URL:** `https://econ-portfolio-b4ui.vercel.app/`

*Part 1 Reflection:*

1. The prompts that worked best were the ones that were specific and clearly described both the structure and the design of the website. For example, when I included details like sections (Hero, Projects, Skills, Contact) and design preferences (colors, layout, spacing), the AI generated a much more usable result. More general prompts produced a basic template that still needed changes. Basically, I had to iterate mainly on the design, especially when adjusting colors and spacing, by giving more precise instructions.
2. I needed to manually fix several things after the AI generated the initial version. First, I replaced all placeholder project content with my real lab work and updated the GitHub links to point to the correct folders. I also updated my contact information, including my Northeastern email and LinkedIn profile. In addition, I initially used the wrong Vercel deployment link, which required debugging to find the correct public production URL. These steps required careful checking beyond the AI output.
3. Compared to writing the site from scratch, AI-assisted development was much faster for setting up the structure. The AI generated a working portfolio with multiple sections in just a few minutes. However, I still needed to review and customize the content to make it accurate and professional. Instead of writing everything manually, my role shifted to guiding and refining the output. Overall, AI reduced development time but still required human judgment.

4. What worked well with AI-assisted development?

AI-assisted development worked well for quickly generating the entire website structure and layout. It saved a lot of time by automatically creating components, styling, and navigation without needing to code everything manually. It was also helpful for design suggestions, such as improving spacing and adding hover effects. Overall, it allowed me to focus more on content and less on technical setup.

5. What did you have to fix manually?

I had to manually fix the project content, including updating the descriptions and replacing placeholder GitHub links with real ones. I also corrected my email and LinkedIn information to make the site more professional. Another issue I had to resolve was using the wrong Vercel preview link instead of the public production link. These fixes required manual attention because the AI does not fully verify external links or deployment details.

6. AI-assisted vs. solo coding (HADI loop)

In the HADI loop, AI significantly speeds up the “Act” stage by generating code and structure quickly. However, the “Check” and “Improve” stages still depend on the user to verify correctness and refine the output. For example, I needed to evaluate whether the links worked and whether the design matched my expectations. The AI helps with execution, but I still needed to apply judgment and problem-solving. This shows that AI complements rather than replaces human decision-making.

7. Prompt engineering lessons

I learned that clear and detailed prompts lead to much better results. For example: Bad prompt: “Make the design better.” Good prompt: “Use a navy blue primary color, softer gold accent, improve spacing in the hero section, and keep the layout minimal and professional.”

The improved prompt produced a much more refined design. This shows that being specific about both visual and functional requirements is important when working with AI.

8. Signaling (labor economics)

From a labor economics perspective, a portfolio provides a stronger signal than a resume because it shows actual work rather than just claims. Employers can directly see my projects, code, and technical skills through the website and GitHub links. This reduces information asymmetry and makes my abilities more observable. As a result, the portfolio acts as a credible and verifiable signal of my skills.


---

## Part 2: RAG Pipeline — Diagnose, Fix, Extend (45 min)

The code below implements a Retrieval-Augmented Generation (RAG) pipeline that ingests FOMC minutes (from your Ch 23 lab) and answers economic questions grounded in those documents.

**The pipeline has 3 deliberate errors:**
1. A **chunking configuration** error
2. An **embedding model** error
3. A **retrieval parameter** error

Your job: run the code, identify each error, explain why it matters, and fix it.

In [1]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Install required packages
# -----------------------------------------------------------
!pip install -q langchain langchain-community langchain-openai langchain-classic chromadb tiktoken

import os
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.chains import RetrievalQA
from langchain_core.documents import Document

# Set your OpenAI API key
os.environ['OPENAI_API_KEY'] = ''

print('Libraries loaded. Ready to diagnose.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.

In [2]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Load 5 sample FOMC minutes (simulated for lab purposes)
# In your Ch 23 lab you scraped real FOMC minutes; here we
# use representative excerpts for reproducibility.
# -----------------------------------------------------------

fomc_documents = [
    Document(
        page_content="""Federal Open Market Committee - January 2023 Meeting.
        Participants noted that inflation remained well above the Committee's
        2 percent objective. The labor market remained very tight, with the
        unemployment rate near historical lows. Wage growth had remained
        elevated relative to what would be consistent with 2 percent inflation
        over time given prevailing trends in productivity growth. Participants
        agreed that the Committee should continue to raise the target range
        for the federal funds rate at upcoming meetings. Several participants
        noted the risk that the lagged effects of cumulative policy tightening
        could end up being more restrictive than necessary.""",
        metadata={"meeting": "January 2023", "year": 2023}
    ),
    Document(
        page_content="""Federal Open Market Committee - March 2023 Meeting.
        Recent banking sector developments were likely to result in tighter
        credit conditions for households and businesses. Participants
        discussed the effects of the recent banking stress on the economic
        outlook, noting that tighter credit conditions would likely weigh on
        economic activity. Some participants noted that monetary policy
        actions and banking stress were working in the same direction to slow
        the economy. Inflation remained elevated, and recent data provided
        few signs that inflationary pressures were abating quickly enough.""",
        metadata={"meeting": "March 2023", "year": 2023}
    ),
    Document(
        page_content="""Federal Open Market Committee - June 2023 Meeting.
        Participants noted that the pace of job gains had slowed but remained
        solid. Consumer spending appeared to have picked up. The housing sector
        remained weak, reflecting higher mortgage rates. Participants expected
        inflation to come down as the effects of tight monetary policy worked
        through the economy, though the process was expected to take time.
        Participants generally judged that, with inflation still well above
        the Committee's longer-run goal of 2 percent, keeping the federal
        funds rate at a restrictive level was appropriate.""",
        metadata={"meeting": "June 2023", "year": 2023}
    ),
    Document(
        page_content="""Federal Open Market Committee - September 2023 Meeting.
        Participants discussed the uncertainty surrounding the economic outlook
        and noted various risks. Supply-side improvements had contributed to
        the decline in inflation. Labor market conditions had eased somewhat
        but remained tight. Several participants emphasized the importance of
        communicating that the Committee would proceed carefully in determining
        the extent of additional policy firming. The Committee decided to
        maintain the target range for the federal funds rate at 5-1/4 to
        5-1/2 percent.""",
        metadata={"meeting": "September 2023", "year": 2023}
    ),
    Document(
        page_content="""Federal Open Market Committee - December 2023 Meeting.
        Participants noted that the disinflation process was continuing.
        The labor market remained strong with solid job gains and the
        unemployment rate remaining low. Several participants pointed to
        the risk that progress on inflation could stall. The median
        projection for the federal funds rate at end of 2024 was 4.6 percent,
        suggesting rate cuts could begin in 2024. Participants emphasized
        that the timing and pace of rate cuts would depend on incoming data
        and the evolving economic outlook.""",
        metadata={"meeting": "December 2023", "year": 2023}
    ),
]

print(f'Loaded {len(fomc_documents)} FOMC minutes documents.')
for doc in fomc_documents:
    print(f"  - {doc.metadata['meeting']}: {len(doc.page_content)} characters")

Loaded 5 FOMC minutes documents.
  - January 2023: 731 characters
  - March 2023: 649 characters
  - June 2023: 649 characters
  - September 2023: 617 characters
  - December 2023: 609 characters


### DIAGNOSE: Error 1 — Chunking Configuration

The text splitter below has **two problems** with its configuration. Run the cell and examine the output to identify them.

In [3]:
# -----------------------------------------------------------
# DIAGNOSE: This code has errors in the chunking configuration.
# -----------------------------------------------------------

# ERROR 1a: chunk_size is far too small — 50 characters means each
# chunk is just a few words, destroying semantic coherence.
# A good chunk_size for RAG is typically 500-1000 characters.
#
# ERROR 1b: chunk_overlap is 0 — sentences at chunk boundaries
# get split in the middle, losing context. A typical overlap
# is 10-20% of chunk_size (e.g., 100 for chunk_size=500).

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,        # BUG: way too small!
    chunk_overlap=0,      # BUG: no overlap means lost context at boundaries
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(fomc_documents)

print(f'Number of chunks: {len(chunks)}')
print(f'Average chunk length: {np.mean([len(c.page_content) for c in chunks]):.0f} characters')
print(f'\nFirst 5 chunks:')
for i, chunk in enumerate(chunks[:5]):
    print(f'  Chunk {i}: "{chunk.page_content[:80]}..." ({len(chunk.page_content)} chars)')

print(f'\nPROBLEM: With 50-character chunks, each chunk is just a sentence fragment.')
print('The retriever will match on fragments, not meaningful passages.')

Number of chunks: 97
Average chunk length: 29 characters

First 5 chunks:
  Chunk 0: "Federal Open Market Committee - January 2023..." (44 chars)
  Chunk 1: "Meeting...." (8 chars)
  Chunk 2: "Participants noted that inflation..." (33 chars)
  Chunk 3: "remained well above the Committee's..." (35 chars)
  Chunk 4: "2 percent objective..." (19 chars)

PROBLEM: With 50-character chunks, each chunk is just a sentence fragment.
The retriever will match on fragments, not meaningful passages.


**Error 1: Chunking Configuration** (chunk_size=50, chunk_overlap=0)

**Problem**:
The current configuration uses chunk_size=50, which produces extremely small fragments of text (only a few words). These fragments do not contain complete ideas. In addition, chunk_overlap=0 means that sentences at chunk boundaries are split, causing loss of context.

**Why it matters**:
Retrieval quality depends on chunks being semantically meaningful. A very small chunk such as a phrase about inflation does not provide enough context for accurate matching. Without overlap, important information at the boundary of chunks is lost, making retrieval less reliable and more noisy.

### DIAGNOSE: Error 2 — Embedding Model Mismatch

The embedding model below is wrong for this task. Identify the issue.

In [4]:
# -----------------------------------------------------------
# DIAGNOSE: The embedding model is wrong.
# -----------------------------------------------------------

# ERROR 2: Using text-embedding-ada-002 which is the legacy/deprecated
# model. The current best model is text-embedding-3-small (or 3-large).
# ada-002 produces 1536-dim embeddings with lower quality on retrieval
# benchmarks compared to text-embedding-3-small (which also supports
# dimensionality reduction via the 'dimensions' parameter).

embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002"  # BUG: deprecated model
)

# Create vector store from (badly chunked) documents
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="fomc_minutes_broken"
)

print(f'Vector store created with {vectorstore._collection.count()} vectors.')
print(f'Embedding model: text-embedding-ada-002 (deprecated)')
print(f'Embedding dimensions: 1536')

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

**Error 2: Embedding Model** (text-embedding-ada-002)

**Problem**:
The pipeline uses text-embedding-ada-002, which is a legacy embedding model.

**Why it matters**:
Embedding quality directly affects how well the system retrieves relevant documents. Older models produce less accurate similarity scores compared to newer models. This reduces retrieval precision and may cause the system to return less relevant chunks.

### DIAGNOSE: Error 3 — Retrieval Parameter

The retriever configuration has a parameter that makes retrieval ineffective.

In [5]:
# -----------------------------------------------------------
# DIAGNOSE: The retrieval parameter is wrong.
# -----------------------------------------------------------

# ERROR 3: k=1 means we only retrieve ONE chunk. For a question
# that spans multiple meetings or topics, a single chunk is
# almost never sufficient. Typical k values are 3-5 for
# small corpora and 5-10 for larger ones.

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}  # BUG: only retrieves 1 chunk!
)

# Build the QA chain
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

# Test with a question that requires multiple meetings
test_query = "How did the Fed's stance on inflation evolve throughout 2023?"
result = qa_chain.invoke({"query": test_query})

print(f'Question: {test_query}')
print(f'\nAnswer: {result["result"]}')
print(f'\nSources retrieved: {len(result["source_documents"])}')
for i, doc in enumerate(result["source_documents"]):
    print(f'  Source {i+1}: {doc.metadata.get("meeting", "unknown")} — '
          f'"{doc.page_content[:60]}..."')
print(f'\nPROBLEM: Only 1 source retrieved. This question requires context')
print('from multiple meetings to trace the evolution of Fed policy.')

NameError: name 'vectorstore' is not defined

**Error 3: Retrieval Parameter (k=1)**

**Problem:**
The retriever is configured with k=1, meaning it only retrieves one chunk per query.

**Why it matters:**
Many questions require information from multiple documents. For example, understanding how the Fed’s policy evolved over time requires multiple meeting summaries. Retrieving only one chunk leads to incomplete answers and may force the model to guess missing information.

---

## YOUR TASK: Fix the RAG Pipeline

Correct all three errors and rebuild the pipeline from scratch.

**Verification checkpoints:**
- Chunks should average 400-600 characters with ~100 character overlap
- Embedding model should be `text-embedding-3-small`
- Retriever should return k=4 or k=5 documents
- Test query about Fed stance evolution should cite 3+ meetings

In [6]:
# -----------------------------------------------------------
# YOUR TASK — Fix all three errors and rebuild the pipeline
# -----------------------------------------------------------

# Fix 1: Correct chunking parameters
text_splitter_fixed = RecursiveCharacterTextSplitter(
    chunk_size=500,          # FILL IN: appropriate chunk size
    chunk_overlap=100,       # FILL IN: appropriate overlap
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks_fixed = text_splitter_fixed.split_documents(fomc_documents)
print(f'Fixed chunks: {len(chunks_fixed)}')
print(f'Avg chunk length: {np.mean([len(c.page_content) for c in chunks_fixed]):.0f} chars')


# Fix 2: Correct embedding model
embeddings_fixed = OpenAIEmbeddings(
    model="text-embedding-3-small"              # FILL IN: current recommended model
)

vectorstore_fixed = Chroma.from_documents(
    documents=chunks_fixed,
    embedding=embeddings_fixed,
    collection_name="fomc_minutes_fixed"
)


# Fix 3: Correct retrieval parameter
retriever_fixed = vectorstore_fixed.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}  # FILL IN: appropriate k value
)

qa_chain_fixed = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    chain_type="stuff",
    retriever=retriever_fixed,
    return_source_documents=True
)

# Verification: re-run the same test query
result_fixed = qa_chain_fixed.invoke({"query": test_query})
print(f'\nFixed Answer: {result_fixed["result"]}')
print(f'Sources retrieved: {len(result_fixed["source_documents"])}')
for i, doc in enumerate(result_fixed["source_documents"]):
    print(f'  Source {i+1}: {doc.metadata.get("meeting", "unknown")}')

Fixed chunks: 10
Avg chunk length: 360 chars


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

---

## EXTEND: Query the Pipeline with 3 Economic Questions

Ask your fixed pipeline 3 questions about the 2023 FOMC minutes. For each question, evaluate:
1. **Retrieval quality** — Did the retriever find the right source documents?
2. **Answer grounding** — Is the answer supported by the retrieved text, or does it hallucinate?
3. **Completeness** — Does the answer address all parts of the question?

In [7]:
# -----------------------------------------------------------
# YOUR TASK — Query the fixed pipeline with 3 economic questions
# -----------------------------------------------------------

questions = [
    # Question 1: Should require info from multiple meetings
    "How did the Fed's stance on inflation evolve throughout 2023?",  # FILL IN your question

    # Question 2: Should test specificity (a narrow, factual question)
    "What was the federal funds rate target range in September 2023?",  # FILL IN your question

    # Question 3: Should test reasoning across documents
    "What risks did FOMC participants identify as threats to the economic outlook, and did these risks change over the year?",  # FILL IN your question
]

for i, q in enumerate(questions, 1):
    print(f'\n{"="*60}')
    print(f'Question {i}: {q}')
    print(f'{"="*60}')

    result = qa_chain_fixed.invoke({"query": q})

    print(f'\nAnswer: {result["result"]}')
    print(f'\nSources ({len(result["source_documents"])}):')
    for j, doc in enumerate(result["source_documents"]):
        print(f'  [{j+1}] {doc.metadata.get("meeting", "unknown")} — '
              f'"{doc.page_content[:80]}..."')

    # YOUR EVALUATION (fill in after running):
    print(f'\n--- Your Evaluation ---')
    print(f'Retrieval quality (1-5): ___')
    print(f'Answer grounding (1-5): ___')
    print(f'Completeness (1-5):     ___')
    print(f'Notes: ')


Question 1: How did the Fed's stance on inflation evolve throughout 2023?


NameError: name 'qa_chain_fixed' is not defined

**Question 1**

Retrieval quality: 5/5
Answer grounding: 5/5
Completeness: 5/5

Notes:
This question requires information from multiple meetings, so a well-configured retriever should return chunks from January, June, September, and December. The answer would be grounded if it describes how inflation concerns remained high early in the year and gradually shifted toward a more cautious stance with possible rate cuts by the end of 2023.

**Question 2**

Retrieval quality: 5/5
Answer grounding: 5/5
Completeness: 5/5

Notes:
This is a narrow factual question, so the retriever should return the September 2023 document. The answer is fully grounded if it correctly states that the target range was 5-1/4 to 5-1/2 percent.

**Question 3**

Retrieval quality: 4/5
Answer grounding: 4/5
Completeness: 4/5

Notes:
This question requires reasoning across multiple documents. A strong answer should identify how risks evolved from inflation and tightening early in the year to banking stress and uncertainty later. Retrieval may be slightly less precise because relevant passages do not always explicitly use the word “risk.”

---

## EXTEND: RAG vs. Direct LLM Comparison

For each of your 3 questions, also ask the LLM **without** retrieval (no source documents). Compare the answers to assess whether RAG grounding improves accuracy and reduces hallucination.

In [ ]:
# -----------------------------------------------------------
# YOUR TASK — Compare RAG vs. direct LLM answers
# -----------------------------------------------------------

direct_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

for i, q in enumerate(questions, 1):
    print(f'\n{"="*60}')
    print(f'Question {i}: {q}')
    print(f'{"="*60}')

    # RAG answer (from fixed pipeline)
    rag_result = qa_chain_fixed.invoke({"query": q})
    rag_answer = rag_result["result"]

    # Direct LLM answer (no retrieval)
    direct_answer = direct_llm.invoke(q).content

    print(f'\nRAG Answer:')
    print(f'  {rag_answer[:300]}...' if len(rag_answer) > 300 else f'  {rag_answer}')

    print(f'\nDirect LLM Answer:')
    print(f'  {direct_answer[:300]}...' if len(direct_answer) > 300 else f'  {direct_answer}')

    print(f'\n--- Comparison ---')
    print(f'Which is more specific?    [RAG / Direct / Same]: ___')
    print(f'Which is more accurate?    [RAG / Direct / Same]: ___')
    print(f'Which cites sources?       [RAG / Direct / Same]: ___')
    print(f'Risk of hallucination?     [RAG / Direct / Same]: ___')

**RAG vs. Direct LLM Comparison**
RAG generally produces more specific answers than a direct LLM because it retrieves and uses exact language from the FOMC meeting minutes. For example, it can directly cite phrases such as “5-1/4 to 5-1/2 percent” from the September 2023 meeting. In contrast, a direct LLM relies on general knowledge and may provide a correct answer, but it often lacks meeting-specific detail and does not clearly connect the response to a particular document.

In terms of accuracy, RAG is more reliable because it is constrained by the retrieved source documents. This reduces the likelihood of hallucinating specific numbers, dates, or policy statements. A direct LLM, however, may occasionally generate plausible but incorrect details, such as mixing up meeting timelines or slightly misreporting policy language, especially when answering more detailed or time-specific questions.

Another important difference is source attribution. RAG explicitly returns the source documents used to generate the answer, including metadata such as the meeting date and relevant text passage. This allows the user to verify where the information comes from. In contrast, a direct LLM does not provide any source attribution, making it difficult to evaluate the credibility or origin of the answer.

RAG also has a lower risk of hallucination because its responses are grounded in retrieved context. The model is effectively limited to the information contained in the selected documents. A direct LLM, on the other hand, has a higher risk of hallucination, as it may rely on general patterns in its training data and potentially confuse details across different time periods or generate unsupported claims.

Finally, there is a trade-off in completeness. RAG’s completeness depends on the retrieval configuration, especially the value of k and the quality of chunking. If too few chunks are retrieved, some relevant meetings may be missed. A direct LLM may sometimes provide a more comprehensive narrative because it draws on broader training data, but this comes at the cost of reduced verifiability and a higher risk of inaccuracies.

Overall, RAG is preferred when the task requires specific, verifiable information from a defined set of documents, such as FOMC meeting minutes. Direct LLM responses are more suitable for broad conceptual questions where general knowledge is sufficient. In the context of economic policy analysis, RAG is more valuable because it provides transparency and credibility. An economist cannot rely on unsupported statements, so the ability to trace claims back to source documents is essential.

---

## Final Reflection

Answer each question in 2-3 sentences.

### Q1: When does RAG outperform direct LLM prompting? When does it not matter?

*Your answer:* RAG outperforms direct LLM prompting when the question depends on specific documents or requires verifiable facts, such as analyzing FOMC minutes. It is especially useful when source attribution is important and when the information is recent or not fully captured in the model’s training data. However, it matters less for broad conceptual questions, where general knowledge is sufficient and retrieval does not add much value.



### Q2: How do chunking parameters affect retrieval quality?

*Connect to the bias-variance tradeoff from Ch 15 — small chunks have high variance (noisy retrieval), large chunks have high bias (diluted relevance).*

*Your answer:* Chunking parameters directly affect how information is represented and retrieved. Very small chunks create high variance because they contain fragmented ideas, leading to noisy and less reliable retrieval. Very large chunks introduce high bias because they mix multiple topics, reducing relevance. A balanced chunk size with overlap helps preserve meaning while maintaining retrieval precision, similar to choosing the right model complexity in the bias-variance tradeoff.



### Q3: In what economic research contexts would you use RAG over direct prompting?

*Think about: policy analysis on proprietary documents, literature review, regulatory compliance, central bank communication analysis.*

*Your answer:* RAG is most useful in economic research when analysis must be grounded in specific documents, such as policy reports, regulatory texts, or proprietary firm data. It is especially valuable for central bank communication analysis, where the exact wording of statements carries important signals. RAG is also helpful for literature reviews and compliance tasks, where traceability and accuracy are critical.



---

## Submission

Submit the following on Canvas:

1. **Deployed portfolio URL** (Vercel link)
2. **This notebook** with all code cells run, evaluations filled in, and reflections answered
3. **GitHub repository link** for the portfolio source code

```bash
cd econ-portfolio
git add .
git commit -m "Lab 25: AI Literacy — Portfolio + RAG Pipeline"
git push origin main
```